# Text Classification Accuracy Evaluation with Gemini

**HELENA · LLM Evaluation for Non-Verifiable Domains**

This notebook evaluates how well LLM-generated natural-language *category descriptions* can be used as zero-shot classifiers, and whether the quality of those descriptions correlates with the semantic separability of the underlying text data.

## High-Level Workflow

| Step | Description |
| ---- | ----------- |
| 1 | Load texts from a Google Drive JSON file organised by category |
| 2 | Split each category 80 % training / 20 % testing |
| 3 | Iteratively generate category descriptions with **Gemini 3.1 Flash Lite** |
| 4 | Classify test texts in randomised batches of 20 using the descriptions |
| 5 | Evaluate accuracy at global and per-category level (charts + tables) |
| 6 | Measure semantic distance preservation with text embeddings |
| 7 | Determine correlation between description accuracy and semantic distance |
| 8 | *(Optional)* Ablation study: **Gemini 3.5 Flash** — all descriptions in one request |

> **Reproducibility** – All LLM responses are persisted to disk.  
> Set `recreate = False` (default) to reuse cached responses from a previous run,  
> or set `recreate = True` to produce a fresh, time-stamped experiment folder  
> **without ever deleting prior results**.


## Requirements

Install the dependencies below once per environment (Colab or local):

```bash
pip install google-generativeai pandas numpy matplotlib seaborn \
            scikit-learn scipy tqdm ipywidgets
```

Set your Gemini API key in the [Configuration](#configuration) cell, or export it beforehand:

```bash
export GEMINI_API_KEY="your-key-here"
```


In [ ]:
# Uncomment to install dependencies in Colab / fresh environments
# !pip install -q google-generativeai pandas numpy matplotlib seaborn scikit-learn scipy tqdm ipywidgets


## Imports


In [ ]:
import os
import re
import json
import math
import time
import random
import hashlib
import datetime
import textwrap
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import cdist, cosine
import google.generativeai as genai
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_palette('husl')
print('Imports OK')


<a id='configuration'></a>
## 0 · Configuration

Edit the values in the `CONFIG` dictionary before running any other cells.

| Key | Type | Description |
| --- | ---- | ----------- |
| `google_drive_path` | `str` | Absolute path to the input JSON inside Google Drive |
| `gemini_api_key` | `str` | Gemini API key (falls back to `GEMINI_API_KEY` env var) |
| `experiment_name` | `str` | Human-readable label for the current experiment |
| `recreate` | `bool` | `True` → new timestamped folder; `False` → reuse cache |
| `description_model` | `str` | Model ID for description generation (Gemini 3.1 Flash Lite) |
| `classification_model` | `str` | Model ID for classification (defaults to same) |
| `embedding_model` | `str` | Model ID for text embeddings |
| `ablation_model` | `str` | Model ID for the ablation study (Gemini 3.5 Flash) |
| `run_ablation` | `bool` | Whether to run the optional ablation section |
| `train_ratio` | `float` | Fraction of data used for training (default 0.8) |
| `random_seed` | `int` | Seed for reproducibility |
| `batch_size` | `int` | Number of texts per classification prompt |
| `description_iterations` | `int` | Refinement rounds for description generation |
| `output_dir` | `str` | Base directory for all experiment artefacts |
| `api_retry_delay` | `float` | Seconds to wait between retries on API errors |
| `api_max_retries` | `int` | Maximum number of API call retries |


In [ ]:
# ─── EXPERIMENT CONFIGURATION ─────────────────────────────────────────────
CONFIG: Dict[str, Any] = {
    # ----- Data ------------------------------------------------------------------
    # Path inside Google Drive to a JSON file with structure:
    #   { "category_name": ["text1", "text2", ...], ... }
    'google_drive_path': '/content/drive/MyDrive/data/categories.json',  # ← CHANGE THIS

    # ----- API -------------------------------------------------------------------
    'gemini_api_key': '',          # Leave empty to use GEMINI_API_KEY env var

    # ----- Experiment ------------------------------------------------------------
    'experiment_name': 'exp_001',  # Label used as directory name under output_dir
    # recreate=False  → reuse cache from experiment_name directory (if it exists)
    # recreate=True   → create a NEW timestamped directory (old data is preserved)
    'recreate': False,

    # ----- Models ----------------------------------------------------------------
    # Gemini 3.1 Flash Lite — used for iterative description generation
    'description_model': 'gemini-3.1-flash-lite-latest',
    # Gemini 3.1 Flash Lite — used for classification (can differ from desc model)
    'classification_model': 'gemini-3.1-flash-lite-latest',
    # Google text embedding model
    'embedding_model': 'models/text-embedding-004',
    # Ablation: Gemini 3.5 Flash — generates all descriptions in a single request
    'ablation_model': 'gemini-3.5-flash-latest',

    # ----- Flags -----------------------------------------------------------------
    'run_ablation': True,          # Set False to skip Section 7

    # ----- Hyperparameters -------------------------------------------------------
    'train_ratio': 0.8,            # 80 % training, 20 % testing
    'random_seed': 42,
    'batch_size': 20,              # Texts per classification prompt
    'description_iterations': 3,   # Refinement rounds for description generation

    # ----- I/O -------------------------------------------------------------------
    'output_dir': './outputs',

    # ----- Resilience ------------------------------------------------------------
    'api_retry_delay': 10.0,       # Seconds between retries
    'api_max_retries': 3,          # Maximum retries per API call
}

# ─── RESOLVE EXPERIMENT DIRECTORY ──────────────────────────────────────────
random.seed(CONFIG['random_seed'])
np.random.seed(CONFIG['random_seed'])

output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

base_exp_dir = output_dir / CONFIG['experiment_name']

if CONFIG['recreate']:
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    exp_dir = output_dir / f"{CONFIG['experiment_name']}_{ts}"
    print(f'Creating new experiment: {exp_dir}')
else:
    exp_dir = base_exp_dir
    print(f'Using/creating experiment directory: {exp_dir}')

exp_dir.mkdir(parents=True, exist_ok=True)

# Subdirectories for different artefact types
cache_dir    = exp_dir / 'cache'        # Raw LLM response JSON files
figures_dir  = exp_dir / 'figures'      # Saved plots
results_dir  = exp_dir / 'results'      # CSV tables
for d in (cache_dir, figures_dir, results_dir):
    d.mkdir(parents=True, exist_ok=True)

# ─── CONFIGURE GEMINI ──────────────────────────────────────────────────────
api_key = CONFIG['gemini_api_key'] or os.environ.get('GEMINI_API_KEY', '')
if not api_key:
    raise EnvironmentError(
        'Gemini API key not found. '
        'Set CONFIG["gemini_api_key"] or export GEMINI_API_KEY.'
    )
genai.configure(api_key=api_key)
print('Gemini configured successfully.')
print(f'  Description model  : {CONFIG["description_model"]}')
print(f'  Classification model: {CONFIG["classification_model"]}')
print(f'  Embedding model    : {CONFIG["embedding_model"]}')


## Utility Functions

### Caching layer

Every LLM call is intercepted by `llm_call()`.  
The request is hashed (model + prompt) and the response stored as a JSON file.  
On subsequent runs with `recreate=False`, the cached response is returned instantly,
avoiding duplicate quota usage and ensuring reproducibility.


In [ ]:
# ─── CACHE UTILITIES ────────────────────────────────────────────────────────

def _cache_key(model: str, prompt: str) -> str:
    """Return a stable hex digest for a (model, prompt) pair."""
    content = json.dumps({'model': model, 'prompt': prompt}, sort_keys=True)
    return hashlib.sha256(content.encode()).hexdigest()[:24]


def _cache_path(key: str, tag: str = '') -> Path:
    """Return the JSON file path for a given cache key."""
    fname = f'{tag}_{key}.json' if tag else f'{key}.json'
    return cache_dir / fname


def load_cache(key: str, tag: str = '') -> Optional[Dict]:
    """Load a cached LLM response, or return None if not found."""
    p = _cache_path(key, tag)
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return None


def save_cache(key: str, data: Dict, tag: str = '') -> None:
    """Persist an LLM response to the cache directory."""
    p = _cache_path(key, tag)
    with open(p, 'w') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


# ─── GEMINI API WRAPPER ──────────────────────────────────────────────────────

def llm_call(
    prompt: str,
    model_name: str,
    tag: str = '',
    temperature: float = 0.2,
    force_refresh: bool = False,
) -> str:
    """
    Send a prompt to a Gemini model and return the text response.

    Parameters
    ----------
    prompt : str
        The full prompt to send.
    model_name : str
        Gemini model identifier (e.g. 'gemini-3.1-flash-lite-latest').
    tag : str
        Optional short label prepended to the cache filename for readability.
    temperature : float
        Sampling temperature passed to the model.
    force_refresh : bool
        When True, bypass cache and call the API even if a response exists.

    Returns
    -------
    str
        The model's text response.
    """
    key = _cache_key(model_name, prompt)
    recreate_flag = CONFIG['recreate'] or force_refresh

    if not recreate_flag:
        cached = load_cache(key, tag)
        if cached is not None:
            return cached['response']

    model = genai.GenerativeModel(
        model_name,
        generation_config=genai.types.GenerationConfig(temperature=temperature),
    )

    for attempt in range(1, CONFIG['api_max_retries'] + 1):
        try:
            result = model.generate_content(prompt)
            response_text = result.text
            break
        except Exception as exc:
            print(f'  API error (attempt {attempt}/{CONFIG["api_max_retries"]}): {exc}')
            if attempt < CONFIG['api_max_retries']:
                time.sleep(CONFIG['api_retry_delay'] * attempt)
            else:
                raise

    save_cache(key, {
        'model': model_name,
        'prompt': prompt,
        'response': response_text,
        'timestamp': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'tag': tag,
    }, tag)

    return response_text


# ─── EMBEDDING WRAPPER ───────────────────────────────────────────────────────

def embed_texts(
    texts: List[str],
    model_name: str,
    tag: str = 'embed',
    batch_size: int = 100,
) -> np.ndarray:
    """
    Generate L2-normalised embeddings for a list of texts.

    Parameters
    ----------
    texts : List[str]
        Input texts to embed.
    model_name : str
        Gemini embedding model identifier.
    tag : str
        Cache tag prefix.
    batch_size : int
        Number of texts per embedding API call.

    Returns
    -------
    np.ndarray
        Shape (N, D) array of unit-norm embeddings.
    """
    cache_key = _cache_key(model_name, json.dumps(texts, sort_keys=True))
    cached = load_cache(cache_key, tag)
    if cached and not CONFIG['recreate']:
        return np.array(cached['embeddings'])

    all_embeddings: List[List[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i: i + batch_size]
        for attempt in range(1, CONFIG['api_max_retries'] + 1):
            try:
                result = genai.embed_content(
                    model=model_name,
                    content=batch,
                    task_type='CLUSTERING',
                )
                all_embeddings.extend(result['embedding'])
                break
            except Exception as exc:
                print(f'  Embedding error (attempt {attempt}): {exc}')
                if attempt < CONFIG['api_max_retries']:
                    time.sleep(CONFIG['api_retry_delay'] * attempt)
                else:
                    raise

    emb_array = np.array(all_embeddings, dtype=np.float32)
    # L2 normalise so cosine distance = Euclidean distance on unit sphere
    norms = np.linalg.norm(emb_array, axis=1, keepdims=True)
    emb_array = emb_array / np.maximum(norms, 1e-10)

    save_cache(cache_key, {'embeddings': emb_array.tolist(), 'texts': texts}, tag)
    return emb_array


# ─── MISC HELPERS ────────────────────────────────────────────────────────────

def wrap(text: str, width: int = 90) -> str:
    """Wrap long strings for readable display."""
    return textwrap.fill(text, width)


def save_figure(fig: plt.Figure, name: str) -> None:
    """Save a matplotlib figure to the figures directory as PNG."""
    path = figures_dir / f'{name}.png'
    fig.savefig(path, bbox_inches='tight')
    print(f'  Saved figure → {path}')


def save_table(df: pd.DataFrame, name: str) -> None:
    """Save a DataFrame to the results directory as CSV."""
    path = results_dir / f'{name}.csv'
    df.to_csv(path)
    print(f'  Saved table  → {path}')


print('Utility functions loaded.')


---
## 1 · Data Loading and Preprocessing

### Expected Input Format

The JSON file at `CONFIG['google_drive_path']` must follow this schema:

```json
{
  "CategoryA": ["text_1", "text_2", ...],
  "CategoryB": ["text_1", "text_2", ...]
}
```

**Type annotations:**

```
InputJSON  = Dict[str, List[str]]
  category (key)    : str   – unique category label
  texts    (value)  : List[str] – raw text samples for that category
```

Each category must contain **at least 5 texts** so that the 80/20 split produces
at least one test example.


In [ ]:
# ─── MOUNT GOOGLE DRIVE (Colab only) ───────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Google Drive mounted.')
except ImportError:
    print('Not running in Colab — skipping Drive mount.')
    print('Make sure CONFIG["google_drive_path"] points to a local file.')


In [ ]:
# ─── LOAD AND VALIDATE DATA ─────────────────────────────────────────────────

def load_dataset(path: str) -> Dict[str, List[str]]:
    """
    Load the category→texts mapping from a JSON file.

    Parameters
    ----------
    path : str
        File system path (absolute or relative) to the input JSON.

    Returns
    -------
    Dict[str, List[str]]
        Mapping from category name to list of raw text strings.

    Raises
    ------
    FileNotFoundError
        If the file does not exist at `path`.
    ValueError
        If the file is not valid JSON or does not match the expected schema.
    """
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f'Input file not found: {path}')

    with open(p) as f:
        raw = json.load(f)

    if not isinstance(raw, dict):
        raise ValueError('JSON root must be an object (dict).')

    cleaned: Dict[str, List[str]] = {}
    for cat, texts in raw.items():
        if not isinstance(texts, list):
            raise ValueError(f'Category "{cat}" must map to a list of strings.')
        texts_str = [str(t).strip() for t in texts if str(t).strip()]
        if len(texts_str) < 5:
            print(f'  WARNING: category "{cat}" has only {len(texts_str)} texts — skipping.')
            continue
        cleaned[str(cat).strip()] = texts_str

    if len(cleaned) < 2:
        raise ValueError('Dataset must contain at least 2 valid categories.')

    return cleaned


dataset: Dict[str, List[str]] = load_dataset(CONFIG['google_drive_path'])
categories: List[str] = sorted(dataset.keys())

print(f'Loaded {len(categories)} categories:')
for cat in categories:
    print(f'  {cat:30s}  {len(dataset[cat]):4d} texts')


In [ ]:
# ─── 80/20 TRAIN / TEST SPLIT ───────────────────────────────────────────────

def split_data(
    dataset: Dict[str, List[str]],
    train_ratio: float,
    seed: int,
) -> Tuple[Dict[str, List[str]], Dict[str, List[str]]]:
    """
    Stratified train/test split preserving per-category proportions.

    Parameters
    ----------
    dataset : Dict[str, List[str]]
        Full dataset mapping category → texts.
    train_ratio : float
        Proportion of texts to place in the training set (e.g. 0.8).
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    train : Dict[str, List[str]]
    test  : Dict[str, List[str]]
    """
    rng = random.Random(seed)
    train: Dict[str, List[str]] = {}
    test:  Dict[str, List[str]] = {}
    for cat, texts in dataset.items():
        shuffled = texts[:]
        rng.shuffle(shuffled)
        n_train = max(1, math.floor(len(shuffled) * train_ratio))
        train[cat] = shuffled[:n_train]
        test[cat]  = shuffled[n_train:]
    return train, test


train_data, test_data = split_data(
    dataset,
    CONFIG['train_ratio'],
    CONFIG['random_seed'],
)

# Summary table
split_summary = pd.DataFrame([
    {
        'Category': cat,
        'Total': len(dataset[cat]),
        'Train': len(train_data[cat]),
        'Test':  len(test_data[cat]),
        'Train %': f'{100 * len(train_data[cat]) / len(dataset[cat]):.0f}',
    }
    for cat in categories
])
print('Data split summary:')
display(split_summary)
save_table(split_summary, 'data_split_summary')


---
## 2 · Category Description Generation (Gemini 3.1 Flash Lite)

For each category, the model is asked to produce a natural-language description
that could serve as a zero-shot classification rule.

The generation is **iterative**: the description is progressively refined over
`CONFIG['description_iterations']` rounds, each round presenting a new random
sample of training texts and asking the model to improve its previous description.

**Why iterative?**  
A single prompt may overfit to surface-level patterns in the first few examples.
Iterating with different samples forces the model to capture features that
generalise across the whole category.

### Prompt Design

```
Round 0 (seed):
  "Here are examples of texts labelled '[Cat]'. Write a concise 2-3 sentence
   description that captures what distinguishes them from other categories."

Round k > 0 (refinement):
  "Here are additional examples of '[Cat]'. Your current description is:
   [prev_description]
   Refine the description to better capture all examples, while remaining
   distinct from other categories: [other_categories]."
```

All intermediate and final descriptions are cached to disk.


In [ ]:
# ─── DESCRIPTION GENERATION ─────────────────────────────────────────────────

def build_seed_prompt(category: str, examples: List[str]) -> str:
    """
    Build the initial (round-0) description prompt for a single category.

    Parameters
    ----------
    category : str
        The category label.
    examples : List[str]
        A sample of training texts from this category.

    Returns
    -------
    str
        Formatted prompt string.
    """
    example_block = '\n'.join(f'  - {t}' for t in examples)
    return (
        f'You are building a text classifier.\n'
        f'Below are example texts belonging to the category **"{category}"**:\n'
        f'{example_block}\n\n'
        f'Write a concise 2–3 sentence description that captures what makes these '
        f'texts belong to "{category}" and distinguishes them from other categories. '
        f'Focus on content, style, topic, and intent. '
        f'Output only the description — no additional commentary.'
    )


def build_refinement_prompt(
    category: str,
    examples: List[str],
    prev_description: str,
    other_categories: List[str],
) -> str:
    """
    Build a refinement prompt that improves an existing description.

    Parameters
    ----------
    category : str
        Current category label.
    examples : List[str]
        New sample of training texts from this category.
    prev_description : str
        The description from the previous iteration.
    other_categories : List[str]
        Labels of all other categories (for contrastive framing).

    Returns
    -------
    str
        Formatted prompt string.
    """
    example_block = '\n'.join(f'  - {t}' for t in examples)
    others = ', '.join(f'"{c}"' for c in other_categories)
    return (
        f'You are refining a text classifier description for the category **"{category}"**.\n\n'
        f'Current description:\n"{prev_description}"\n\n'
        f'Here are additional training examples for "{category}":\n'
        f'{example_block}\n\n'
        f'The other categories in the classifier are: {others}.\n\n'
        f'Refine the description so it (1) covers the new examples, '
        f'(2) remains clearly distinct from the other categories, and '
        f'(3) stays concise (2–4 sentences). '
        f'Output only the improved description — no additional commentary.'
    )


def generate_descriptions(
    train_data: Dict[str, List[str]],
    model_name: str,
    n_iterations: int,
    seed: int,
    tag_prefix: str = 'desc',
) -> Dict[str, str]:
    """
    Iteratively generate and refine a description for every category.

    Parameters
    ----------
    train_data : Dict[str, List[str]]
        Training texts per category.
    model_name : str
        Gemini model identifier.
    n_iterations : int
        Number of refinement rounds (total calls per category = n_iterations + 1).
    seed : int
        RNG seed for example sampling.
    tag_prefix : str
        Cache tag prefix (useful to separate main vs ablation caches).

    Returns
    -------
    Dict[str, str]
        Mapping from category name to final description string.
    """
    rng = random.Random(seed)
    cats = sorted(train_data.keys())
    descriptions: Dict[str, str] = {}

    # How many examples to show per iteration
    def n_examples(texts: List[str]) -> int:
        return min(8, max(3, len(texts) // 4))

    for cat in tqdm(cats, desc='Generating descriptions'):
        texts = train_data[cat][:]
        rng.shuffle(texts)
        ne = n_examples(texts)

        # Round 0: seed description
        examples = texts[:ne]
        prompt = build_seed_prompt(cat, examples)
        desc = llm_call(prompt, model_name, tag=f'{tag_prefix}_seed_{cat}')
        desc = desc.strip()

        # Rounds 1..n_iterations: refinement
        for it in range(1, n_iterations + 1):
            # Sample a fresh batch of examples (with replacement if needed)
            start = (it * ne) % max(1, len(texts))
            new_examples = (texts + texts)[start: start + ne]
            other_cats = [c for c in cats if c != cat]
            prompt = build_refinement_prompt(cat, new_examples, desc, other_cats)
            desc = llm_call(
                prompt, model_name,
                tag=f'{tag_prefix}_iter{it}_{cat}'
            ).strip()

        descriptions[cat] = desc

    return descriptions


print('Generating category descriptions...')
descriptions: Dict[str, str] = generate_descriptions(
    train_data,
    model_name=CONFIG['description_model'],
    n_iterations=CONFIG['description_iterations'],
    seed=CONFIG['random_seed'],
    tag_prefix='main_desc',
)
print('Done.')


In [ ]:
# ─── DISPLAY GENERATED DESCRIPTIONS ────────────────────────────────────────
print('=' * 80)
for cat, desc in descriptions.items():
    print(f'\n[{cat}]')
    print(wrap(desc))
    print('-' * 80)

# Save descriptions to disk
desc_path = results_dir / 'descriptions_main.json'
with open(desc_path, 'w') as f:
    json.dump(descriptions, f, indent=2, ensure_ascii=False)
print(f'\nDescriptions saved to {desc_path}')


---
## 3 · Text Classification Using Generated Descriptions

All test texts across categories are pooled together, **shuffled**, and presented
to the model in batches of `CONFIG['batch_size']` (default 20).

The model receives the full set of category descriptions and must assign each
text to exactly one category.

### Prompt Template

```
You are a text classifier. Classify each numbered text into one of the following
categories using their descriptions:

CategoryA: <description_A>
CategoryB: <description_B>
...

Return a JSON array where each element is the category name for the
corresponding text (same order as inputs).

Texts:
1. <text_1>
2. <text_2>
...
```

Responses are parsed and any unparseable batches are retried once.


In [ ]:
# ─── CLASSIFICATION HELPERS ─────────────────────────────────────────────────

def build_classification_prompt(
    descriptions: Dict[str, str],
    texts: List[str],
) -> str:
    """
    Build a prompt that asks the model to classify a batch of texts.

    Parameters
    ----------
    descriptions : Dict[str, str]
        Category-to-description mapping.
    texts : List[str]
        The texts to classify.

    Returns
    -------
    str
        Formatted prompt.
    """
    desc_block = '\n'.join(
        f'  {cat}: {desc}' for cat, desc in sorted(descriptions.items())
    )
    text_block = '\n'.join(f'  {i+1}. {t}' for i, t in enumerate(texts))
    valid_cats = ', '.join(f'"{c}"' for c in sorted(descriptions))

    return (
        f'You are a text classifier. '
        f'Classify each numbered text into exactly one of the following categories.\n\n'
        f'Category descriptions:\n{desc_block}\n\n'
        f'Instructions:\n'
        f'  - Return a **JSON array** of {len(texts)} strings.\n'
        f'  - Each string must be one of: {valid_cats}.\n'
        f'  - Preserve the original order.\n'
        f'  - Output ONLY the JSON array, nothing else.\n\n'
        f'Texts to classify:\n{text_block}'
    )


def parse_classification_response(
    response: str,
    expected_len: int,
    valid_labels: List[str],
) -> Optional[List[str]]:
    """
    Parse the JSON array returned by the classification prompt.

    Parameters
    ----------
    response : str
        Raw model response string.
    expected_len : int
        Expected number of classifications.
    valid_labels : List[str]
        Set of valid category labels.

    Returns
    -------
    List[str] or None
        Parsed list of category labels, or None if parsing failed.
    """
    # Strip markdown code fences if present
    cleaned = re.sub(r'```(?:json)?\s*', '', response).strip().rstrip('`')
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        # Try extracting the first JSON array
        m = re.search(r'\[.*\]', cleaned, re.DOTALL)
        if not m:
            return None
        try:
            parsed = json.loads(m.group())
        except json.JSONDecodeError:
            return None

    if not isinstance(parsed, list) or len(parsed) != expected_len:
        return None

    # Normalise labels (case-insensitive match)
    valid_lower = {v.lower(): v for v in valid_labels}
    result = []
    for item in parsed:
        label = str(item).strip()
        matched = valid_lower.get(label.lower())
        if matched is None:
            return None   # Unknown label — reject the whole batch
        result.append(matched)

    return result


def classify_all(
    test_data: Dict[str, List[str]],
    descriptions: Dict[str, str],
    model_name: str,
    batch_size: int,
    seed: int,
    tag_prefix: str = 'classify',
) -> Tuple[List[str], List[str]]:
    """
    Classify all test texts and return true and predicted label lists.

    Parameters
    ----------
    test_data : Dict[str, List[str]]
        Test texts organised by true category.
    descriptions : Dict[str, str]
        Category descriptions to use for classification.
    model_name : str
        Gemini model identifier.
    batch_size : int
        Number of texts per classification prompt.
    seed : int
        RNG seed for batch shuffling.
    tag_prefix : str
        Cache tag prefix.

    Returns
    -------
    y_true : List[str]  — ground-truth category labels
    y_pred : List[str]  — model-predicted category labels
    """
    # Flatten and shuffle
    all_items: List[Tuple[str, str]] = []
    for cat, texts in test_data.items():
        for t in texts:
            all_items.append((cat, t))

    rng = random.Random(seed)
    rng.shuffle(all_items)

    y_true: List[str] = []
    y_pred: List[str] = []
    valid_labels = sorted(descriptions.keys())

    for start in tqdm(range(0, len(all_items), batch_size), desc='Classifying batches'):
        batch = all_items[start: start + batch_size]
        batch_texts = [t for _, t in batch]
        batch_labels = [c for c, _ in batch]

        prompt = build_classification_prompt(descriptions, batch_texts)

        for attempt in range(1, CONFIG['api_max_retries'] + 1):
            response = llm_call(
                prompt, model_name,
                tag=f'{tag_prefix}_b{start}',
                force_refresh=(attempt > 1),
            )
            preds = parse_classification_response(response, len(batch_texts), valid_labels)
            if preds is not None:
                break
            print(f'    Parse failed (attempt {attempt}), retrying...')
            if attempt == CONFIG['api_max_retries']:
                # Fallback: assign first category to all (worst-case)
                print(f'    Fallback: assigning "{valid_labels[0]}" to all {len(batch_texts)} texts.')
                preds = [valid_labels[0]] * len(batch_texts)

        y_true.extend(batch_labels)
        y_pred.extend(preds)

    return y_true, y_pred


print('Starting classification...')
y_true, y_pred = classify_all(
    test_data,
    descriptions,
    model_name=CONFIG['classification_model'],
    batch_size=CONFIG['batch_size'],
    seed=CONFIG['random_seed'],
    tag_prefix='main_classify',
)
print(f'Classified {len(y_true)} test texts.')


---
## 4 · Performance Evaluation

We report accuracy, precision, recall and F1 at two levels:

* **Global** – macro-averaged across all categories.
* **Per-category** – individual scores for each class.

Visualisations include:
* Confusion matrix (normalised)
* Bar charts of per-category F1 and accuracy
* Summary table (saved as CSV)


In [ ]:
# ─── GLOBAL METRICS ─────────────────────────────────────────────────────────

def compute_metrics(
    y_true: List[str],
    y_pred: List[str],
    labels: List[str],
) -> Tuple[Dict[str, float], pd.DataFrame]:
    """
    Compute global and per-category classification metrics.

    Parameters
    ----------
    y_true : List[str]
        Ground-truth labels.
    y_pred : List[str]
        Predicted labels.
    labels : List[str]
        Ordered list of category labels.

    Returns
    -------
    global_metrics : Dict[str, float]
        Keys: 'accuracy', 'macro_precision', 'macro_recall', 'macro_f1'
    per_cat_df : pd.DataFrame
        Per-category precision, recall, F1, support, and accuracy.
    """
    global_metrics = {
        'accuracy':        accuracy_score(y_true, y_pred),
        'macro_precision': precision_score(y_true, y_pred, labels=labels, average='macro', zero_division=0),
        'macro_recall':    recall_score(y_true, y_pred, labels=labels, average='macro', zero_division=0),
        'macro_f1':        f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0),
    }

    report = classification_report(
        y_true, y_pred, labels=labels, output_dict=True, zero_division=0
    )
    rows = []
    for cat in labels:
        d = report[cat]
        # Per-category accuracy = TP / (TP + FN) = recall for that class
        rows.append({
            'Category':  cat,
            'Precision': d['precision'],
            'Recall':    d['recall'],
            'F1':        d['f1-score'],
            'Support':   int(d['support']),
            'Accuracy':  d['recall'],  # = recall for a single class
        })
    per_cat_df = pd.DataFrame(rows).set_index('Category')
    return global_metrics, per_cat_df


global_metrics, per_cat_df = compute_metrics(y_true, y_pred, categories)

print('─' * 60)
print('GLOBAL METRICS')
print('─' * 60)
for k, v in global_metrics.items():
    print(f'  {k:20s}: {v:.4f}')

print('\nPER-CATEGORY METRICS')
display(per_cat_df.round(4))

save_table(per_cat_df, 'per_category_metrics_main')


In [ ]:
# ─── CONFUSION MATRIX ───────────────────────────────────────────────────────

cm = confusion_matrix(y_true, y_pred, labels=categories, normalize='true')
cm_df = pd.DataFrame(cm, index=categories, columns=categories)

fig, ax = plt.subplots(figsize=(max(6, len(categories)), max(5, len(categories) - 1)))
sns.heatmap(
    cm_df, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5,
    cbar_kws={'label': 'Recall'},
    ax=ax,
)
ax.set_title('Normalised Confusion Matrix (row = true, col = predicted)', pad=12)
ax.set_xlabel('Predicted Category')
ax.set_ylabel('True Category')
plt.tight_layout()
save_figure(fig, 'confusion_matrix_main')
plt.show()


In [ ]:
# ─── PER-CATEGORY BAR CHARTS ────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 score per category
sorted_f1 = per_cat_df['F1'].sort_values()
axes[0].barh(sorted_f1.index, sorted_f1.values, color=sns.color_palette('husl', len(sorted_f1)))
axes[0].axvline(global_metrics['macro_f1'], ls='--', color='gray', label=f'Macro F1 = {global_metrics["macro_f1"]:.3f}')
axes[0].set_xlabel('F1 Score')
axes[0].set_title('Per-Category F1 Score')
axes[0].legend()
axes[0].set_xlim(0, 1)

# Accuracy (recall) per category
sorted_acc = per_cat_df['Accuracy'].sort_values()
axes[1].barh(sorted_acc.index, sorted_acc.values, color=sns.color_palette('husl', len(sorted_acc)))
axes[1].axvline(global_metrics['accuracy'], ls='--', color='gray', label=f'Overall Acc = {global_metrics["accuracy"]:.3f}')
axes[1].set_xlabel('Accuracy (Recall)')
axes[1].set_title('Per-Category Accuracy')
axes[1].legend()
axes[1].set_xlim(0, 1)

fig.suptitle('Classification Performance by Category', fontsize=13, y=1.01)
plt.tight_layout()
save_figure(fig, 'per_category_performance_main')
plt.show()


In [ ]:
# ─── SUMMARY STATS TABLE ────────────────────────────────────────────────────

summary_df = pd.DataFrame([global_metrics]).T.rename(columns={0: 'Value'})
summary_df['Value'] = summary_df['Value'].map(lambda x: f'{x:.4f}')
print('Global Performance Summary (Main Model)')
display(summary_df)
save_table(summary_df, 'global_metrics_main')


---
## 5 · Semantic Distance Analysis

We use **text embeddings** (Google `text-embedding-004`) to measure the geometric
structure of each category in the embedding space.

### Metrics Computed

| Metric | Symbol | Formula | Interpretation |
| ------ | ------ | ------- | -------------- |
| Intra-category spread | $\sigma_c$ | Mean cosine distance from texts to category centroid | Lower = tighter cluster |
| Inter-category distance | $d_{ij}$ | Cosine distance between category centroids $i$ and $j$ | Higher = better separation |
| Silhouette score | $s_c$ | $(b_c - a_c) / \max(a_c, b_c)$ | Higher = better separation |
| Separation Preservation Score | **SPS** | $\bar{d}_{inter} / \bar{\sigma}_{intra}$ | Overall separability ratio |

### Separation Preservation Score (SPS)

The **SPS** is our custom metric for how well category boundaries are preserved
in embedding space.  It is the ratio of the mean inter-category centroid distance
to the mean intra-category spread:

$$
\text{SPS} = \frac{\displaystyle\frac{1}{\binom{C}{2}} \sum_{i < j} d(\mu_i, \mu_j)}{\displaystyle\frac{1}{C} \sum_c \sigma_c}
$$

A higher SPS indicates that categories are far apart relative to their internal
spread — making them easier to distinguish by description alone.

We also compute a **per-category SPS** as the mean distance to all *other*
category centroids, normalised by that category's intra-spread — this will be
used in the correlation analysis.


In [ ]:
# ─── GENERATE EMBEDDINGS ────────────────────────────────────────────────────

# Embed training texts (used to compute centroids and spreads)
# We also embed test texts for completeness

all_train_texts: List[str] = []
all_train_labels: List[str] = []
for cat in categories:
    for t in train_data[cat]:
        all_train_texts.append(t)
        all_train_labels.append(cat)

print(f'Embedding {len(all_train_texts)} training texts...')
train_embeddings = embed_texts(
    all_train_texts,
    CONFIG['embedding_model'],
    tag='emb_train',
)
print(f'Embedding shape: {train_embeddings.shape}')


In [ ]:
# ─── COMPUTE SEMANTIC DISTANCE METRICS ──────────────────────────────────────

def compute_semantic_metrics(
    embeddings: np.ndarray,
    labels: List[str],
    categories: List[str],
) -> Tuple[Dict[str, float], Dict[str, float], Dict[str, float], float, pd.DataFrame]:
    """
    Compute intra/inter-category semantic distance metrics.

    Parameters
    ----------
    embeddings : np.ndarray
        Shape (N, D) — L2-normalised text embeddings.
    labels : List[str]
        Category label for each embedding (length N).
    categories : List[str]
        Ordered list of category names.

    Returns
    -------
    centroids        : Dict[str, np.ndarray]  (1-D arrays of shape D)
    intra_spread     : Dict[str, float]       per-category mean cosine distance to centroid
    per_cat_sps      : Dict[str, float]       per-category SPS
    global_sps       : float                  global Separation Preservation Score
    distance_matrix  : pd.DataFrame           C×C inter-centroid cosine distances
    """
    labels_arr = np.array(labels)

    # ── Centroids ────────────────────────────────────────────────────────────
    centroids: Dict[str, np.ndarray] = {}
    for cat in categories:
        mask = labels_arr == cat
        c_emb = embeddings[mask]
        centroid = c_emb.mean(axis=0)
        norm = np.linalg.norm(centroid)
        centroids[cat] = centroid / max(norm, 1e-10)

    # ── Intra-category spread (mean cosine distance to centroid) ─────────────
    intra_spread: Dict[str, float] = {}
    for cat in categories:
        mask = labels_arr == cat
        c_emb = embeddings[mask]
        # Cosine distance = 1 - cosine similarity = 1 - dot product (unit vectors)
        sims = c_emb @ centroids[cat]  # dot products (since all unit-norm)
        dists = 1.0 - sims
        intra_spread[cat] = float(dists.mean())

    # ── Inter-category centroid distances ────────────────────────────────────
    centroid_matrix = np.stack([centroids[c] for c in categories])
    # Cosine distance between unit vectors: dist = 1 - dot
    inter_matrix = 1.0 - centroid_matrix @ centroid_matrix.T
    np.fill_diagonal(inter_matrix, 0.0)
    distance_matrix = pd.DataFrame(inter_matrix, index=categories, columns=categories)

    # ── Silhouette-style per-category SPS ────────────────────────────────────
    per_cat_sps: Dict[str, float] = {}
    for i, cat in enumerate(categories):
        other_dists = [inter_matrix[i, j] for j, c in enumerate(categories) if c != cat]
        mean_inter = np.mean(other_dists) if other_dists else 0.0
        spread = intra_spread[cat]
        per_cat_sps[cat] = float(mean_inter / max(spread, 1e-10))

    # ── Global SPS ───────────────────────────────────────────────────────────
    n = len(categories)
    if n < 2:
        global_sps = 0.0
    else:
        pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
        mean_inter = np.mean([inter_matrix[i, j] for i, j in pairs])
        mean_intra = np.mean(list(intra_spread.values()))
        global_sps = float(mean_inter / max(mean_intra, 1e-10))

    return centroids, intra_spread, per_cat_sps, global_sps, distance_matrix


centroids, intra_spread, per_cat_sps, global_sps, distance_matrix = compute_semantic_metrics(
    train_embeddings, all_train_labels, categories
)

print(f'Global Separation Preservation Score (SPS): {global_sps:.4f}')
print('\nPer-category SPS and intra-spread:')
sem_df = pd.DataFrame({
    'Intra-Spread': intra_spread,
    'SPS': per_cat_sps,
}).sort_values('SPS', ascending=False)
display(sem_df.round(4))
save_table(sem_df, 'semantic_metrics_main')


In [ ]:
# ─── VISUALISE SEMANTIC DISTANCES ───────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of inter-category centroid distances
sns.heatmap(
    distance_matrix, annot=True, fmt='.3f', cmap='YlOrRd_r',
    linewidths=0.5, ax=axes[0],
    cbar_kws={'label': 'Cosine Distance'},
)
axes[0].set_title('Inter-Category Centroid Distances')

# Bar chart: SPS per category
sem_sorted = sem_df.sort_values('SPS')
axes[1].barh(
    sem_sorted.index, sem_sorted['SPS'],
    color=sns.color_palette('rocket', len(sem_sorted)),
)
axes[1].axvline(global_sps, ls='--', color='gray',
                label=f'Global SPS = {global_sps:.3f}')
axes[1].set_xlabel('Separation Preservation Score (SPS)')
axes[1].set_title('Per-Category SPS')
axes[1].legend()

fig.suptitle('Semantic Distance Analysis', fontsize=13, y=1.01)
plt.tight_layout()
save_figure(fig, 'semantic_distances_main')
plt.show()


In [ ]:
# ─── PCA VISUALISATION OF EMBEDDINGS ────────────────────────────────────────

pca = PCA(n_components=2, random_state=CONFIG['random_seed'])
emb_2d = pca.fit_transform(train_embeddings)

fig, ax = plt.subplots(figsize=(9, 7))
palette = sns.color_palette('husl', len(categories))

for idx, cat in enumerate(categories):
    mask = np.array(all_train_labels) == cat
    ax.scatter(
        emb_2d[mask, 0], emb_2d[mask, 1],
        label=cat, alpha=0.55, s=35, color=palette[idx],
    )
    # Plot centroid
    cx, cy = emb_2d[mask, 0].mean(), emb_2d[mask, 1].mean()
    ax.scatter(cx, cy, marker='*', s=250, color=palette[idx],
               edgecolors='black', linewidths=0.8, zorder=5)

var_explained = pca.explained_variance_ratio_
ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}%)')
ax.set_title('PCA of Training Embeddings (★ = centroid)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
save_figure(fig, 'pca_embeddings_main')
plt.show()


---
## 6 · Correlation: Description Accuracy vs. Semantic Distance

**Research Question**: *Is there a statistically significant correlation between
a category's classification accuracy and how well-separated that category is
in embedding space (measured by its per-category SPS)?*

### Hypothesis

* **H₀**: There is no correlation between per-category accuracy and per-category SPS.
* **H₁**: Higher SPS (better semantic separation) predicts higher classification accuracy.

### Method

For each category we have two scalars:
* `accuracy_c`  – fraction of test texts from category *c* correctly classified
* `sps_c`       – Separation Preservation Score for category *c*

We compute both **Pearson** (linear) and **Spearman** (rank) correlations,
and report *p*-values to assess statistical significance.


In [ ]:
# ─── BUILD CORRELATION TABLE ─────────────────────────────────────────────────

corr_df = pd.DataFrame({
    'Category':  categories,
    'Accuracy':  [per_cat_df.loc[c, 'Accuracy'] for c in categories],
    'F1':        [per_cat_df.loc[c, 'F1']       for c in categories],
    'SPS':       [per_cat_sps[c]                 for c in categories],
    'IntraSpread': [intra_spread[c]              for c in categories],
}).set_index('Category')

display(corr_df.round(4))
save_table(corr_df, 'correlation_table_main')


In [ ]:
# ─── STATISTICAL CORRELATION TESTS ──────────────────────────────────────────

def correlation_report(
    x: np.ndarray,
    y: np.ndarray,
    x_name: str,
    y_name: str,
) -> Dict[str, Any]:
    """
    Compute Pearson and Spearman correlations and return a summary dict.

    Parameters
    ----------
    x, y : np.ndarray
        Paired numeric arrays.
    x_name, y_name : str
        Display names for the two variables.

    Returns
    -------
    Dict with keys: pearson_r, pearson_p, spearman_r, spearman_p
    """
    p_r, p_p = pearsonr(x, y)
    s_r, s_p = spearmanr(x, y)
    result = {
        'x': x_name, 'y': y_name,
        'pearson_r': p_r,  'pearson_p': p_p,
        'spearman_r': s_r, 'spearman_p': s_p,
        'n': len(x),
    }
    sig_p = '*' if p_p < 0.05 else ''
    sig_s = '*' if s_p < 0.05 else ''
    print(f'  {x_name} vs {y_name}  (n={len(x)})')
    print(f'    Pearson  r={p_r:+.4f}  p={p_p:.4f} {sig_p}')
    print(f'    Spearman r={s_r:+.4f}  p={s_p:.4f} {sig_s}')
    return result


print('─' * 60)
print('CORRELATION: Accuracy vs SPS')
print('─' * 60)
corr_acc_sps = correlation_report(
    corr_df['SPS'].values, corr_df['Accuracy'].values,
    'SPS', 'Accuracy'
)

print('\nCORRELATION: F1 vs SPS')
print('─' * 60)
corr_f1_sps = correlation_report(
    corr_df['SPS'].values, corr_df['F1'].values,
    'SPS', 'F1'
)

print('\nCORRELATION: Accuracy vs IntraSpread (inverse)')
print('─' * 60)
corr_acc_intra = correlation_report(
    corr_df['IntraSpread'].values, corr_df['Accuracy'].values,
    'IntraSpread', 'Accuracy'
)

all_corr = pd.DataFrame([corr_acc_sps, corr_f1_sps, corr_acc_intra])
save_table(all_corr, 'correlation_statistics_main')


In [ ]:
# ─── SCATTER PLOTS ───────────────────────────────────────────────────────────

import numpy.polynomial.polynomial as nppoly

def scatter_with_regression(
    ax: plt.Axes,
    x: np.ndarray,
    y: np.ndarray,
    labels: List[str],
    x_name: str,
    y_name: str,
    pearson_r: float,
    pearson_p: float,
    color: str = 'steelblue',
) -> None:
    """Draw a scatter plot with a linear regression line and annotation labels."""
    ax.scatter(x, y, color=color, s=80, zorder=3, edgecolors='black', linewidths=0.6)
    for xi, yi, lab in zip(x, y, labels):
        ax.annotate(
            lab, (xi, yi),
            textcoords='offset points', xytext=(6, 4),
            fontsize=8, color='#333333',
        )

    # Regression line
    if len(x) > 1:
        coefs = nppoly.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, nppoly.polyval(x_line, coefs),
                'r--', linewidth=1.5, alpha=0.7)

    sig = '(p<0.05 *)' if pearson_p < 0.05 else '(p≥0.05)'
    ax.set_title(f'{y_name} vs {x_name}\nPearson r={pearson_r:+.3f} {sig}', fontsize=10)
    ax.set_xlabel(x_name)
    ax.set_ylabel(y_name)
    ax.grid(True, alpha=0.3)


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

scatter_with_regression(
    axes[0], corr_df['SPS'].values, corr_df['Accuracy'].values,
    list(corr_df.index), 'SPS', 'Accuracy',
    corr_acc_sps['pearson_r'], corr_acc_sps['pearson_p'],
)
scatter_with_regression(
    axes[1], corr_df['SPS'].values, corr_df['F1'].values,
    list(corr_df.index), 'SPS', 'F1',
    corr_f1_sps['pearson_r'], corr_f1_sps['pearson_p'],
)
scatter_with_regression(
    axes[2], corr_df['IntraSpread'].values, corr_df['Accuracy'].values,
    list(corr_df.index), 'IntraSpread', 'Accuracy',
    corr_acc_intra['pearson_r'], corr_acc_intra['pearson_p'],
    color='darkorange',
)

fig.suptitle(
    'Correlation: Classification Accuracy vs. Semantic Separation\n(Main Model)',
    fontsize=13, y=1.03,
)
plt.tight_layout()
save_figure(fig, 'correlation_scatter_main')
plt.show()


In [ ]:
# ─── CORRELATION VERDICT ────────────────────────────────────────────────────

def verdict(corr_r: float, corr_p: float, x_name: str, y_name: str) -> str:
    """
    Generate a plain-English verdict for a correlation result.

    Parameters
    ----------
    corr_r : float   Pearson correlation coefficient.
    corr_p : float   Two-tailed p-value.
    x_name : str     Name of the predictor variable.
    y_name : str     Name of the outcome variable.

    Returns
    -------
    str  Human-readable verdict.
    """
    strength = (
        'strong'   if abs(corr_r) >= 0.7 else
        'moderate' if abs(corr_r) >= 0.4 else
        'weak'
    )
    direction = 'positive' if corr_r >= 0 else 'negative'
    significance = 'statistically significant (p < 0.05)' if corr_p < 0.05 else 'not statistically significant (p ≥ 0.05)'

    return (
        f'There is a {strength} {direction} correlation between {x_name} and {y_name} '
        f'(Pearson r = {corr_r:+.3f}, p = {corr_p:.4f}), which is {significance}. '
        f'{"This supports H₁." if corr_p < 0.05 else "H₀ cannot be rejected."}'
    )


print('=' * 80)
print('CORRELATION VERDICTS')
print('=' * 80)
print()
print('1. SPS → Accuracy')
print('   ' + verdict(corr_acc_sps['pearson_r'], corr_acc_sps['pearson_p'], 'SPS', 'Accuracy'))
print()
print('2. SPS → F1')
print('   ' + verdict(corr_f1_sps['pearson_r'], corr_f1_sps['pearson_p'], 'SPS', 'F1'))
print()
print('3. IntraSpread → Accuracy')
print('   ' + verdict(corr_acc_intra['pearson_r'], corr_acc_intra['pearson_p'], 'IntraSpread', 'Accuracy'))


### Interpretation

The verdicts above are automatically generated from the statistical tests.
Key points to consider when reading them:

* A **positive SPS → Accuracy** correlation would confirm that categories which are
  semantically well-separated (far from each other relative to their own spread)
  are also easier for the LLM to classify using descriptions.

* A **negative IntraSpread → Accuracy** correlation would mean that tightly clustered
  categories (low spread) yield higher classification accuracy.

* Note that with a small number of categories, the sample size (*n*) is limited,
  which reduces statistical power.  A non-significant result does not mean there
  is no relationship — it may simply mean we need more categories.

* Run the notebook with more categories (≥ 20) for robust conclusions.


---
## 7 · Ablation Study (Optional): Gemini 3.5 Flash — Single-Request Descriptions

This section compares our iterative description approach against a **single-prompt**
strategy using the more capable **Gemini 3.5 Flash** model.

### Key Differences

| Dimension | Main (§2–§6) | Ablation (§7) |
| --------- | ----------- | ------------- |
| Model | Gemini 3.1 Flash Lite | Gemini 3.5 Flash |
| Calls per description | `n_iterations + 1` per category | 1 total call for all categories |
| Examples used | Progressive sampling | All training examples |

### Single-Request Prompt

```
You are building a text classifier with the following categories:
[CatA, CatB, ...]

For each category you will receive training examples. Write a 2-4 sentence
description for EACH category that distinguishes it from the others.

Return a JSON object: { "CategoryName": "description", ... }

--- CategoryA ---
example 1
example 2 ...
--- CategoryB ---
...
```

This cell is guarded by `CONFIG['run_ablation']`.


In [ ]:
if not CONFIG['run_ablation']:
    print('Ablation study skipped (CONFIG["run_ablation"] = False).')
else:
    # ── BUILD SINGLE-REQUEST PROMPT ────────────────────────────────────────
    def build_ablation_prompt(train_data: Dict[str, List[str]]) -> str:
        """
        Build a single mega-prompt that requests descriptions for ALL categories.

        Parameters
        ----------
        train_data : Dict[str, List[str]]
            All training texts per category.

        Returns
        -------
        str  Formatted prompt.
        """
        cats = sorted(train_data.keys())
        cat_list = ', '.join(f'"{c}"' for c in cats)
        sections = []
        for cat in cats:
            examples = '\n'.join(f'  - {t}' for t in train_data[cat])
            sections.append(f'--- {cat} ---\n{examples}')
        body = '\n\n'.join(sections)
        return (
            f'You are building a text classifier for the following categories: {cat_list}.\n\n'
            f'Below are ALL training examples for each category.\n\n'
            f'{body}\n\n'
            f'For each category, write a concise 2-4 sentence description that '
            f'distinguishes it from the others.\n'
            f'Return ONLY a JSON object mapping category name to description:\n'
            f'{{ "CategoryName": "description", ... }}'
        )


    # ── CALL ABLATION MODEL ────────────────────────────────────────────────
    print(f'Running ablation with {CONFIG["ablation_model"]}...')
    ablation_prompt = build_ablation_prompt(train_data)
    ablation_raw = llm_call(
        ablation_prompt,
        model_name=CONFIG['ablation_model'],
        tag='ablation_descriptions',
        temperature=0.2,
    )

    # ── PARSE RESPONSE ────────────────────────────────────────────────────
    def parse_ablation_descriptions(
        raw: str,
        expected_cats: List[str],
    ) -> Optional[Dict[str, str]]:
        """
        Parse the JSON object of category descriptions from the ablation response.

        Parameters
        ----------
        raw : str
            Raw model response.
        expected_cats : List[str]
            The category names we expect to find in the JSON.

        Returns
        -------
        Dict[str, str] or None  Category-to-description mapping, or None on failure.
        """
        cleaned = re.sub(r'```(?:json)?\s*', '', raw).strip().rstrip('`')
        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError:
            m = re.search(r'\{.*\}', cleaned, re.DOTALL)
            if not m:
                return None
            try:
                parsed = json.loads(m.group())
            except json.JSONDecodeError:
                return None
        if not isinstance(parsed, dict):
            return None
        # Accept case-insensitive matches
        lower_map = {c.lower(): c for c in expected_cats}
        result = {}
        for k, v in parsed.items():
            norm = lower_map.get(str(k).strip().lower())
            if norm:
                result[norm] = str(v).strip()
        return result if len(result) == len(expected_cats) else None


    ablation_descriptions = parse_ablation_descriptions(ablation_raw, categories)
    if ablation_descriptions is None:
        print('WARNING: Could not parse ablation descriptions. Raw response:')
        print(ablation_raw[:500])
    else:
        print(f'Parsed {len(ablation_descriptions)} ablation descriptions.')
        # Save
        abl_desc_path = results_dir / 'descriptions_ablation.json'
        with open(abl_desc_path, 'w') as f:
            json.dump(ablation_descriptions, f, indent=2, ensure_ascii=False)
        print(f'Saved to {abl_desc_path}')


In [ ]:
if CONFIG['run_ablation'] and 'ablation_descriptions' in dir() and ablation_descriptions:
    # ── CLASSIFY WITH ABLATION DESCRIPTIONS ──────────────────────────────
    print('Classifying with ablation descriptions...')
    y_true_abl, y_pred_abl = classify_all(
        test_data,
        ablation_descriptions,
        model_name=CONFIG['ablation_model'],
        batch_size=CONFIG['batch_size'],
        seed=CONFIG['random_seed'],
        tag_prefix='ablation_classify',
    )

    # ── EVALUATE ABLATION PERFORMANCE ────────────────────────────────────
    global_metrics_abl, per_cat_df_abl = compute_metrics(y_true_abl, y_pred_abl, categories)

    print('\nAblation Global Metrics:')
    for k, v in global_metrics_abl.items():
        print(f'  {k:20s}: {v:.4f}')

    save_table(per_cat_df_abl, 'per_category_metrics_ablation')
    print('\nAblation Per-Category Metrics:')
    display(per_cat_df_abl.round(4))


In [ ]:
if CONFIG['run_ablation'] and 'global_metrics_abl' in dir():
    # ── SIDE-BY-SIDE COMPARISON ───────────────────────────────────────────
    metric_keys = list(global_metrics.keys())
    comparison_df = pd.DataFrame({
        'Main (Gemini 3.1 FL, iterative)':  [global_metrics[k]     for k in metric_keys],
        'Ablation (Gemini 3.5 F, single)':  [global_metrics_abl[k] for k in metric_keys],
    }, index=metric_keys).round(4)
    comparison_df['Delta'] = (
        comparison_df['Ablation (Gemini 3.5 F, single)'] -
        comparison_df['Main (Gemini 3.1 FL, iterative)']
    ).round(4)

    print('Global Metric Comparison:')
    display(comparison_df)
    save_table(comparison_df, 'comparison_main_vs_ablation')

    # ── PER-CATEGORY COMPARISON BAR CHART ────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    x = np.arange(len(categories))
    width = 0.35

    for ax, metric in zip(axes, ['F1', 'Accuracy']):
        bars_main = per_cat_df[metric].reindex(categories).values
        bars_abl  = per_cat_df_abl[metric].reindex(categories).values
        ax.bar(x - width/2, bars_main, width, label='Main (iterative, Gemini 3.1 FL)')
        ax.bar(x + width/2, bars_abl,  width, label='Ablation (single, Gemini 3.5 F)')
        ax.set_xticks(x)
        ax.set_xticklabels(categories, rotation=40, ha='right', fontsize=9)
        ax.set_ylabel(metric)
        ax.set_title(f'Per-Category {metric}: Main vs Ablation')
        ax.legend(fontsize=9)
        ax.set_ylim(0, 1.05)
        ax.grid(axis='y', alpha=0.3)

    fig.suptitle('Main vs Ablation Study', fontsize=13, y=1.01)
    plt.tight_layout()
    save_figure(fig, 'comparison_main_vs_ablation')
    plt.show()


In [ ]:
if CONFIG['run_ablation'] and 'global_metrics_abl' in dir():
    # ── ABLATION CONFUSION MATRIX ─────────────────────────────────────────
    cm_abl = confusion_matrix(y_true_abl, y_pred_abl, labels=categories, normalize='true')
    cm_abl_df = pd.DataFrame(cm_abl, index=categories, columns=categories)

    fig, axes = plt.subplots(1, 2, figsize=(max(10, len(categories)*2), max(4, len(categories)-1)))

    for ax, cm_data, title in [
        (axes[0], pd.DataFrame(confusion_matrix(y_true, y_pred, labels=categories, normalize='true'),
                               index=categories, columns=categories), 'Main'),
        (axes[1], cm_abl_df, 'Ablation'),
    ]:
        sns.heatmap(cm_data, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5,
                    cbar_kws={'label': 'Recall'}, ax=ax)
        ax.set_title(f'Confusion Matrix: {title}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')

    plt.tight_layout()
    save_figure(fig, 'confusion_matrix_comparison')
    plt.show()


---
## Summary

The cells above execute the full HELENA text-classification evaluation pipeline.
All artefacts — LLM responses, metrics CSVs, and figures — are stored under:

```
outputs/
└── <experiment_name>/
    ├── cache/      ← JSON-cached LLM responses (keyed by SHA-256 of prompt)
    ├── figures/    ← PNG plots
    └── results/    ← CSV tables and JSON description files
```

### Re-running

* **Reuse cache**: set `CONFIG['recreate'] = False` — all LLM calls are served from disk.
* **Fresh run**: set `CONFIG['recreate'] = True` — a new timestamped directory is created;
  previous experiments are **never overwritten or deleted**.

### Key Outputs

| File | Description |
| ---- | ----------- |
| `results/descriptions_main.json` | Final category descriptions (main model) |
| `results/per_category_metrics_main.csv` | Per-category classification metrics |
| `results/semantic_metrics_main.csv` | Per-category SPS and intra-spread |
| `results/correlation_statistics_main.csv` | Pearson/Spearman correlation statistics |
| `results/comparison_main_vs_ablation.csv` | Main vs ablation global metric comparison |
| `figures/confusion_matrix_main.png` | Normalised confusion matrix |
| `figures/pca_embeddings_main.png` | PCA of category embedding space |
| `figures/correlation_scatter_main.png` | Accuracy vs SPS scatter plots |
| `figures/comparison_main_vs_ablation.png` | Side-by-side bar chart |


In [ ]:
# ─── SAVE EXPERIMENT CONFIG ─────────────────────────────────────────────────
# Serialise config (excluding sensitive key)
safe_config = {k: v for k, v in CONFIG.items() if k != 'gemini_api_key'}
safe_config['experiment_dir'] = str(exp_dir)
safe_config['timestamp'] = datetime.datetime.now(datetime.timezone.utc).isoformat()

config_path = exp_dir / 'experiment_config.json'
with open(config_path, 'w') as f:
    json.dump(safe_config, f, indent=2)

print(f'Experiment config saved to {config_path}')
print('\nAll done! ✓')
